# 🚀 DeepFake Detection System — Cloud GPU Training
This notebook trains the **Hybrid CNN + Vision Transformer + Frequency Branch** model on a free GPU (Kaggle / Google Colab T4).

---

### 💡 PRO-TIP for Kaggle (Never lose your files):
Instead of keeping the browser tab open for an hour:
1. Click **"Save Version"** (top right corner).
2. Select **"Save & Run All (Commit)"**.
3. Click **"Save"**.

Kaggle will execute the entire notebook in the cloud even if you close your laptop or disconnect. When done, your `detector_best.pt` file is permanently saved in the **Output** tab!

### Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Clone or Pull Latest Code & Install Dependencies

In [ ]:
import os

# If repo already exists, pull latest updates; otherwise clone fresh
if os.path.exists('DeepFake-Detection-System'):
    %cd DeepFake-Detection-System
    !git pull origin main
else:
    !git clone https://github.com/Adi-015/DeepFake-Detection-System.git
    %cd DeepFake-Detection-System

# Install required packages
!pip install -q -r requirements.txt
print("✅ Code and dependencies ready!")

### Step 3: Setup the 140k Real & Fake Faces Dataset

- **On Kaggle:** Make sure you clicked **"+ Add Input"** → searched **`140k Real and Fake Faces`** (by xhlulu) → added it.
- **On Colab:** The script below automatically downloads the dataset via `kagglehub`.

In [ ]:
import os
from pathlib import Path

# Check Kaggle dataset path
kaggle_path = Path('/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake')
if kaggle_path.exists():
    DATA_DIR = str(kaggle_path)
    print(f"✅ Found Kaggle dataset at: {DATA_DIR}")
else:
    # Fallback for Colab
    !pip install -q kagglehub
    import kagglehub
    path = kagglehub.dataset_download('xhlulu/140k-real-and-fake-faces')
    DATA_DIR = os.path.join(path, 'real_vs_fake', 'real-vs-fake')
    print(f"✅ Dataset downloaded to: {DATA_DIR}")

# Ensure output directory is outside repo root in Kaggle so it shows directly in Output
if os.path.exists('/kaggle/working'):
    SAVE_DIR = '/kaggle/working/saved_model'
else:
    SAVE_DIR = './saved_model'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📁 Checkpoints will be saved to: {SAVE_DIR}")

### Step 4: Run Training
- Backbone: `efficientnet_b3`
- Epochs: 10
- Batch size: 64
- Automatic Mixed Precision (AMP) enabled

In [ ]:
!python model/train.py \
  --data_dir "$DATA_DIR" \
  --save_dir "$SAVE_DIR" \
  --backbone efficientnet_b3 \
  --epochs 10 \
  --batch_size 64 \
  --lr 1e-4 \
  --num_workers 2

### Step 5: Download the Best Model Checkpoint
Run this cell immediately when training finishes to get a direct browser download link for `detector_best.pt`!

In [ ]:
import os
from pathlib import Path
from IPython.display import FileLink, display

# Candidate paths for the checkpoint
candidates = [
    Path(SAVE_DIR) / 'detector_best.pt',
    Path('/kaggle/working/saved_model/detector_best.pt'),
    Path('./saved_model/detector_best.pt'),
    Path('/kaggle/working/DeepFake-Detection-System/saved_model/detector_best.pt')
]

found_path = None
for c in candidates:
    if c.exists():
        found_path = c
        break

if not found_path:
    # Search /kaggle/working recursively
    for root, dirs, files in os.walk('/kaggle/working'):
        if 'detector_best.pt' in files:
            found_path = Path(root) / 'detector_best.pt'
            break

if found_path and found_path.exists():
    size_mb = found_path.stat().st_size / (1024 * 1024)
    print(f"🎉 SUCCESS! Found checkpoint ({size_mb:.2f} MB) at: {found_path}")
    print("⬇️ Click the blue link below to download directly to your computer:")
    display(FileLink(str(found_path)))

    # Also download confusion matrix if present
    cm_path = found_path.parent / 'confusion_matrix.png'
    if cm_path.exists():
        display(FileLink(str(cm_path)))

    # Google Colab automatic download trigger
    try:
        from google.colab import files
        files.download(str(found_path))
    except ImportError:
        pass
else:
    print("❌ Checkpoint not found yet. Make sure Step 4 completed successfully.")